# 61 · AI & RAG — the LLM gateway (LiteLLM)

**One OpenAI-compatible front door in front of the whole model fleet.** Where the
notebooks before this reached *stores* — SQL engines, vector indexes, a feature store —
this one reaches *models*. Every hosted and local LLM the lab can call sits behind a
single `/v1` endpoint that speaks the **OpenAI API**, so any OpenAI-compatible client
(the `openai` Python SDK, LangChain, `curl`) talks to the whole fleet without knowing which
provider is underneath.

That indirection is the whole point of a gateway:

> **The client sends a *use-case* name; the gateway decides *which model* serves it, and
> falls over to the next one if that provider is down, rate-limited, or out of credit.**

### What the gateway buys you

- **One API, many providers.** Groq, Anthropic, Gemini, OpenRouter, DeepSeek, Cerebras,
  xAI, Perplexity — plus local Ollama models on the GPU box — all reached through the same
  OpenAI-shaped `/v1/chat/completions`. Swap the provider without touching client code.
- **Use-case aliases (`wl-*`).** Instead of pinning a client to `claude-haiku-4-5` or
  `gpt-oss-120b`, you call `wl-coding`, `wl-rag`, `wl-speed`. Each alias is a **primary model
  plus a server-side fallback chain**; the gateway walks the chain on network / 5xx / 429 /
  timeout, transparently. Re-point an alias once and every client follows.
- **Cost & observability.** The hosted lanes egress through Bifrost, so every funded call is
  recorded with cost / tokens / latency and attributed to a virtual key — the gateway is also
  the spend valve.

### What this notebook does

It connects to LiteLLM as an ordinary OpenAI client, **discovers** the live model list (and
separates the `wl-*` aliases from the raw passthroughs and wildcards), runs a **chat
completion** and a **streamed** one, renders the **routing map** (each alias to its purpose),
and sends the **same prompt to two different aliases** so you can watch routing change the
model that answers.

> **Read-only, throughout.** Every call here is an inference call — `models.list()` and
> `chat.completions.create(...)`. We never touch LiteLLM's admin API (`/key`, `/model`,
> `/user`): this is the client's view of the gateway, not the operator's. And we never
> *assume* which aliases exist — the live `/v1/models` list is authoritative; the routing-map
> doc only supplies the human-readable *purpose* of each alias we actually find.

## Setup — the OpenAI client

The only dependency is the **`openai`** Python SDK — the gateway is OpenAI-compatible, so
the official client drives it unchanged. It is not in the singleuser base image (which ships
`polars`, `s3fs`, `pyarrow`, `duckdb`, `fastavro`), so we install it here. `polars` — used to
render every result frame, exactly as in the query/vector notebooks — already ships in the
image.

In [1]:
%pip install -q openai


[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: /tmp/nbvenv/bin/python -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


## Connect — env-driven, in-cluster default

Connection is **env-driven**, the same pattern the store notebooks use. The committed default
is the **in-cluster** service URL
(`http://litellm.weyland.svc.cluster.local:4000/v1`); a validation run overrides
`LITELLM_BASE_URL` via the environment (e.g. to a NodePort) **without editing the notebook**.
So the notebook never captures the resolved address — the connection is proven by the **model
list it returns**, not by echoing the endpoint.

LiteLLM here is **master-key-only** (no per-user virtual keys), so the bearer token is read
once from `LITELLM_MASTER_KEY` and never committed. The `openai` client takes the gateway's
`/v1` as its `base_url` and the master key as its `api_key`; from there it is just the OpenAI
SDK.

In [2]:
import os
from openai import OpenAI
import polars as pl

# committed default = in-cluster service DNS; a validation run overrides LITELLM_BASE_URL via env
BASE_URL = os.environ.get("LITELLM_BASE_URL", "http://litellm.weyland.svc.cluster.local:4000/v1")
API_KEY  = os.environ["LITELLM_MASTER_KEY"]   # master-key-only gateway; never committed

client = OpenAI(base_url=BASE_URL, api_key=API_KEY)

# prove the connection by WHAT IT RETURNS, never by echoing the resolved endpoint
model_ids = sorted(m.id for m in client.models.list().data)
print(f"connected to the LiteLLM gateway - {len(model_ids)} models served") 

connected to the LiteLLM gateway - 235 models served


## The catalogue — aliases, passthroughs, wildcards

`GET /v1/models` returns everything the gateway will route, and it comes in three shapes:

- **`wl-*` use-case aliases** — the front door most clients should use. Each is a primary
  model plus a fallback chain; the client names the *job*, not the model.
- **wildcards** (`anthropic/*`, `gemini/*`, `openrouter/*`) — pass-through routes that let a
  client name *any* model a provider offers, on demand.
- **named passthroughs** — everything else: specific provider models exposed directly
  (`claude-haiku`, `gemini-flash`, `deepseek-chat`, provider-prefixed ids, …) for when you
  deliberately want one exact model and no fallback.

We classify the live list into those three buckets rather than hard-coding them — so the table
reflects whatever the gateway is actually serving today.

In [3]:
aliases   = [m for m in model_ids if m.startswith("wl-")]
wildcards = [m for m in model_ids if m.endswith("/*")]
passthru  = [m for m in model_ids if not m.startswith("wl-") and not m.endswith("/*")]

print(f"wl-* aliases : {len(aliases)}")
print(f"wildcards    : {len(wildcards)}  ->", wildcards)
print(f"passthroughs : {len(passthru)}  (e.g.", passthru[:6], "...)")

# the aliases are the part worth naming in full - this is the gateway's public menu
pl.DataFrame({"wl-alias": aliases})

wl-* aliases : 11
wildcards    : 3  -> ['anthropic/*', 'gemini/*', 'openrouter/*']
passthroughs : 221  (e.g. ['anthropic/claude-3-7-sonnet-20250219', 'anthropic/claude-3-haiku-20240307', 'anthropic/claude-3-opus-20240229', 'anthropic/claude-4-opus-20250514', 'anthropic/claude-4-sonnet-20250514', 'anthropic/claude-fable-5'] ...)


wl-alias
str
"""wl-agentic"""
"""wl-big-oss"""
"""wl-coding"""
"""wl-default"""
"""wl-judge"""
…
"""wl-rag"""
"""wl-reason"""
"""wl-search"""


## A chat completion — call the *job*, see the *model*

The core call: `chat.completions.create(model="wl-speed", ...)`. We ask for the lowest-latency
lane and a short answer. Two things are worth pulling out of the response:

- **The `model` field echoes the alias**, not the underlying model — the gateway deliberately
  hides which provider served the call, so the client stays decoupled from it.
- **The routing is exposed in the response *headers*.** The `openai` client hands them back
  via `.with_raw_response`, so we can see which model group was hit and where it egressed —
  the proof that an alias really did resolve to a concrete provider.

`wl-speed`'s primary is a reasoning model, so we give it enough `max_tokens` to finish its
internal reasoning *and* emit a visible answer (too small a budget spends everything on
reasoning and returns empty content).

In [4]:
raw = client.chat.completions.with_raw_response.create(
    model="wl-speed",
    max_tokens=400,
    messages=[
        {"role": "system", "content": "You are concise."},
        {"role": "user", "content": "In one sentence, what is an LLM gateway?"},
    ],
)
resp = raw.parse()   # the parsed ChatCompletion

print("answer     :", resp.choices[0].message.content.strip())
print("model field:", resp.model, "(the alias, not the underlying model)")
print("usage      :", resp.usage.prompt_tokens, "prompt +",
      resp.usage.completion_tokens, "completion =", resp.usage.total_tokens, "tokens")

# routing is in the headers - which model group, where it egressed, which provider/model resolved
h = raw.headers
routing = {
    "model_group":      h.get("x-litellm-model-group"),
    "routed_api_base":  h.get("x-litellm-model-api-base"),
    "provider":         h.get("llm_provider-x-bifrost-provider"),
    "resolved_model":   h.get("llm_provider-x-bifrost-resolved-model"),
    "attempted_fallbacks": h.get("x-litellm-attempted-fallbacks"),
}
pl.DataFrame([{"header": k, "value": v} for k, v in routing.items() if v is not None])

answer     : An LLM gateway is an API‑level interface that mediates, routes, and manages access to one or multiple large language models, handling tasks like authentication, request routing, throttling, and response formatting.
model field: wl-speed (the alias, not the underlying model)
usage      : 89 prompt + 96 completion = 185 tokens


header,value
str,str
"""model_group""","""wl-speed"""
"""routed_api_base""","""http://bifrost.weyland.svc.clu…"
"""provider""","""groq"""
"""resolved_model""","""openai/gpt-oss-120b"""
"""attempted_fallbacks""","""0"""


## Streaming — tokens as they arrive

The same call with `stream=True` returns an iterator of chunks instead of one response;
each chunk carries a slice of the answer in `choices[0].delta.content`. This is what a chat
UI consumes to render text as it is generated. We **accumulate** the deltas and print only the
final assembled text — not the raw chunk dump — plus a count of how many chunks arrived, which
is the proof the response really streamed rather than landing in one piece.

In [5]:
stream = client.chat.completions.create(
    model="wl-speed",
    max_tokens=400,
    stream=True,
    messages=[{"role": "user", "content": "List three benefits of an LLM gateway. Be brief."}],
)

chunks = 0
pieces = []
for chunk in stream:
    delta = chunk.choices[0].delta.content
    if delta:
        pieces.append(delta)
        chunks += 1

answer = "".join(pieces).strip()
print(f"streamed in {chunks} content chunks:\n")
print(answer)

streamed in 84 content chunks:

1. **Unified Access** – Provides a single, consistent API for multiple language models, simplifying integration across different providers.  
2. **Scalable Management** – Handles load balancing, rate‑limiting, and model routing automatically, enabling smooth scaling as demand grows.  
3. **Enhanced Security & Governance** – Centralizes authentication, logging, and policy enforcement, helping organizations meet compliance and data‑privacy requirements.


## The routing map — which alias is for what

An alias only helps if you know which job it names. The lab publishes the full map (primary +
fallback chain per alias) at `docs/llm-routing-map.html`; below we pair **the aliases that are
live right now** with the purpose each is designed for. The list is discovered from the
gateway; the purposes are the alias meanings from that routing-map doc.

The chains behind each alias follow one philosophy: **primary = cheapest capable source**
(free or local where the data is private), every chain **ends in a free, always-on rung** so
it can never hard-fail, and a dry paid provider (402/429) is *skipped*, not treated as an
error. The `wl-rag` / `wl-reason` / `wl-judge` lanes stay **local on the GPU box** (private,
$0); the rest egress to hosted providers through Bifrost.

In [6]:
# purposes = the alias meanings from docs/llm-routing-map.html; the LIST comes from the live gateway
PURPOSE = {
    "wl-default":   "General-purpose chat",
    "wl-speed":     "Lowest latency",
    "wl-coding":    "Code generation & edits",
    "wl-agentic":   "Tool orchestration / agents",
    "wl-rag":       "Retrieval-grounded generation (private, local)",
    "wl-reason":    "Deep reasoning (local)",
    "wl-judge":     "Eval scoring / LLM-as-judge (local)",
    "wl-judge-oss": "Eval scoring / LLM-as-judge, OSS variant (local)",
    "wl-search":    "Web-grounded answers",
    "wl-big-oss":   "Big frontier OSS model",
    "wl-tts":       "Text-to-speech (/v1/audio/speech)",
}

# build the table from the LIVE aliases; flag any live alias the doc has not described yet
pl.DataFrame(
    [{"wl-alias": a, "purpose": PURPOSE.get(a, "(live - not yet in the routing-map doc)")}
     for a in aliases]
)

wl-alias,purpose
str,str
"""wl-agentic""","""Tool orchestration / agents"""
"""wl-big-oss""","""Big frontier OSS model"""
"""wl-coding""","""Code generation & edits"""
"""wl-default""","""General-purpose chat"""
"""wl-judge""","""Eval scoring / LLM-as-judge (l…"
…,…
"""wl-rag""","""Retrieval-grounded generation …"
"""wl-reason""","""Deep reasoning (local)"""
"""wl-search""","""Web-grounded answers"""


## Same prompt, two aliases — routing changes the model

The clearest demonstration that an alias *is* a routing decision: send **one prompt** to two
different aliases and watch a different model answer each. We use a small logic riddle and
compare:

- **`wl-speed`** — the low-latency lane, a fast hosted model reached in well under a second.
- **`wl-reason`** — the deep-reasoning lane, a **local** model on the GPU box; slower to
  respond, private, and $0.

Both should reach the same correct answer; what differs is the lane that got there — same
client code, same prompt, two different models, chosen entirely by the alias.

In [7]:
import time

riddle = "A farmer has 17 sheep. All but 9 run away. How many are left? Answer in one short sentence."

def ask(alias, prompt, max_tokens=300):
    t0 = time.time()
    raw = client.chat.completions.with_raw_response.create(
        model=alias, max_tokens=max_tokens,
        messages=[{"role": "user", "content": prompt}],
    )
    r = raw.parse()
    # hosted lanes egress through Bifrost and report the resolved provider model in a header;
    # local lanes stay direct to the GPU box, so that header is simply absent
    resolved = raw.headers.get("llm_provider-x-bifrost-resolved-model")
    routed = resolved if resolved else "local (direct, not via Bifrost)"
    return {
        "alias":      alias,
        "routed_to":  routed,
        "latency_s":  round(time.time() - t0, 2),
        "answer":     r.choices[0].message.content.strip(),
    }

rows = [ask("wl-speed", riddle), ask("wl-reason", riddle)]
pl.DataFrame(rows)

alias,routed_to,latency_s,answer
str,str,f64,str
"""wl-speed""","""openai/gpt-oss-120b""",0.48,"""There are 9 sheep left."""
"""wl-reason""","""local (direct, not via Bifrost…",12.78,"""9"""


## When to reach for the gateway — and which one

The lab runs **two** OpenAI-compatible gateways plus the option of calling providers directly.
They are not redundant — each owns a lane:

| you want to… | reach for | why |
|--------------|-----------|-----|
| programmatic / agentic calls, tool orchestration, RAG retrieval, the widest model menu with fallback | **LiteLLM** (this notebook) | the agentic/programmatic front door: `wl-*` aliases, server-side fallback chains, MCP tool payloads pass through cleanly |
| governed chat / eval traffic with guardrails and a hard budget | **MLflow AI Gateway** (nb `51` area) | a *governed* front door — LLM-judge safety/PII guardrails, a global spend cap, endpoint-name routing; it even fronts LiteLLM as one included option |
| one exact model, no routing, no gateway | **call the provider directly** | when you must pin a specific model/version and want no fallback or indirection at all |

**Reach for LiteLLM when the caller is code, not a person** — an agent, a pipeline step, a RAG
retriever, an eval judge. You get:

- **Decoupling** — name the *job* (`wl-coding`, `wl-rag`), not the model; re-point the alias
  once and every caller follows.
- **Resilience** — the fallback chain means a provider outage, rate-limit, or dry credit is a
  transparent hop to the next rung, never a hard failure.
- **A single client** — the OpenAI SDK you already know, pointed at one `/v1`, reaching the
  whole fleet: hosted providers and local GPU models alike.

**Prefer the MLflow AI Gateway when the traffic needs governance** — human-facing chat or eval
runs that must pass safety/PII guardrails and stay under budget. **Call a provider directly**
only when you truly need one pinned model and no gateway at all. For everything the platform's
own agents and pipelines do, LiteLLM is the default door.